# 06. LR 행 정규화 × C 탐색

## 배경
- 05에서 LR C=0.1 → CV 0.3818 / LB 0.2739, CV−LB 차이 약 0.108
- train과 test의 변이 비율이 다를 가능성이 있음 (test 변이 칸 비율이 더 높아 보임)
  → 단, test 분포는 진단 용도로만 보고 튜닝 근거로 쓰지 않음

## 가설
- H1: 행 정규화(l1, l2)를 하면 모델이 "변이 개수"에 덜 민감해져 일반화가 좋아진다
- H0: 행 정규화는 CV를 개선하지 않는다

## 검증 방법
- 공통 CV (StratifiedKFold 5, seed 42), Macro F1
- 정규화: none / l2 / l1  ×  C 5개씩 (정규화 시 스케일이 작아지므로 C 범위를 다르게 설정)
- 판단 기준: 현재 최고 CV 0.3818 대비 +0.01 이상이면 채택 후 제출
- 행 단위 연산이므로 Data Leakage 아님

In [1]:
import sys; sys.path.append('..')
import time, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from src.config import SEED
from src.data import load_data, get_gene_cols
from src.features import to_binary
from src.validation import seed_everything, encode_target, run_cv, save_submission

seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
X, X_test = to_binary(train, genes).astype('float32'), to_binary(test, genes).astype('float32')
y, le = encode_target(train)

In [2]:
def make(norm, C):
    pre = {'none': FunctionTransformer(),          # 그대로 (05와 같은 조건)
           'l2':   Normalizer('l2'),               # 행 벡터 길이를 1로
           'l1':   Normalizer('l1')}[norm]         # 변이 개수로 나누기
    return lambda: make_pipeline(pre, LogisticRegression(
        C=C, max_iter=3000, class_weight='balanced', random_state=SEED))

grid = {'none': [0.05, 0.07, 0.1, 0.15, 0.2],
        'l2':   [0.3, 1, 3, 10, 30],
        'l1':   [3, 10, 30, 100, 300]}

rows = []
for norm, Cs in grid.items():
    for C in Cs:
        t = time.time()
        r = run_cv(make(norm, C), X, y, verbose=False)
        rows.append(dict(norm=norm, C=C, cv=r['cv_mean'], std=r['cv_std'], oof=r['oof_score'], sec=round(time.time()-t)))
        print(rows[-1])
res = pd.DataFrame(rows).sort_values('cv', ascending=False)
res

{'norm': 'none', 'C': 0.05, 'cv': 0.3838190233315374, 'std': 0.007927521435212194, 'oof': 0.3850511311538694, 'sec': 3}
{'norm': 'none', 'C': 0.07, 'cv': 0.38207606250214365, 'std': 0.005282130212518587, 'oof': 0.38310638784560097, 'sec': 3}
{'norm': 'none', 'C': 0.1, 'cv': 0.38178147354864483, 'std': 0.0064609576320192015, 'oof': 0.3830018495807861, 'sec': 3}
{'norm': 'none', 'C': 0.15, 'cv': 0.3772894201246083, 'std': 0.005318187307467219, 'oof': 0.37875434862050683, 'sec': 3}
{'norm': 'none', 'C': 0.2, 'cv': 0.3735974667428152, 'std': 0.00943168514605811, 'oof': 0.3756603431308276, 'sec': 4}
{'norm': 'l2', 'C': 0.3, 'cv': 0.328821500726325, 'std': 0.011207769278666183, 'oof': 0.32884849772527236, 'sec': 3}
{'norm': 'l2', 'C': 1, 'cv': 0.36451101266948316, 'std': 0.008158873272095132, 'oof': 0.3661129736238823, 'sec': 5}
{'norm': 'l2', 'C': 3, 'cv': 0.3636330531903192, 'std': 0.007429898366473913, 'oof': 0.3651690204567759, 'sec': 7}
{'norm': 'l2', 'C': 10, 'cv': 0.34352377694845415,

,norm,C,cv,std,oof,sec
0,none,0.05,0.383819,0.007928,0.385051,3
1,none,0.07,0.382076,0.005282,0.383106,3
2,none,0.10,0.381781,0.006461,0.383002,3
3,none,0.15,0.377289,0.005318,0.378754,3
4,none,0.20,0.373597,0.009432,0.375660,4
6,l2,1.00,0.364511,0.008159,0.366113,5
7,l2,3.00,0.363633,0.007430,0.365169,7
8,l2,10.00,0.343524,0.017414,0.345836,10
9,l2,30.00,0.329768,0.014917,0.331920,13
5,l2,0.30,0.328822,0.011208,0.328848,3


In [3]:
rows2 = []
for C in [0.01, 0.02, 0.03, 0.04]:
    r = run_cv(make('none', C), X, y, verbose=False)
    rows2.append(dict(norm='none', C=C, cv=r['cv_mean'], std=r['cv_std'], oof=r['oof_score']))
    print(rows2[-1])

{'norm': 'none', 'C': 0.01, 'cv': 0.3429997314809371, 'std': 0.012506636895829261, 'oof': 0.34351949700253925}
{'norm': 'none', 'C': 0.02, 'cv': 0.3682177725236711, 'std': 0.011897358260343235, 'oof': 0.3684199278283059}
{'norm': 'none', 'C': 0.03, 'cv': 0.37390090531226, 'std': 0.011515731202489367, 'oof': 0.37585547463617647}
{'norm': 'none', 'C': 0.04, 'cv': 0.3806138257138071, 'std': 0.006978559525357871, 'oof': 0.38168898594810263}


## 결과

| 정규화 | 최적 C | CV (± std) | 0.3818 대비 |
| :--- | :-: | :-: | :-: |
| none | 0.05 | 0.3838 ± 0.0079 | +0.002 |
| l2 | 1 | 0.3645 ± 0.0082 | −0.017 |
| l1 | 10 | 0.3057 ± 0.0072 | −0.076 |

(작은 C 추가 탐색: C=0.01~0.04 → 최고 0.3806 (C=0.04), 최적은 0.05로 확정 )

## 결론
- **H1 기각.** 행 정규화는 CV를 낮춤. 정규화가 강할수록(l1 > l2) 더 크게 하락
- 해석: 정규화로 "변이 개수" 정보가 사라지면 성능이 떨어짐
  → 변이 개수 자체가 암종을 구분하는 유용한 신호로 보임
  (예: 변이가 거의 없는 THYM, LAML, THCA)
- none + C=0.05는 +0.002로 기준(+0.01) 미달 → 제출하지 않음

## 다음 실험으로 이어지는 가설
- H2: 변이 개수, 침묵 변이 수, 절단 변이(nonsense, frameshift) 수 같은
  행 단위 피처를 추가하면 CV가 오른다 → 07 노트북에서 검증